### Prediction with Group-based Splitting

This notebook demonstrates the process of predicting input values using pre-trained models from `forward_model_new.ipynb` and `diffusion_model_new.ipynb`. The goal is to refine predictions to achieve desired target values.

#### Key Features:
- **Pre-split Data**: Uses pre-split `combined_augmented_train.xlsx` and `val.xlsx` to prevent data leakage
- **Consistent Scalers**: Uses `scaler_x_new.pkl` and `scaler_y_new.pkl` fitted on training set
- **Updated Models**: Uses `EnhancedDiffusionModel` and `ImprovedRegressionModel` with latest weights

#### Process:
1. **Model Loading**: Load pre-trained forward and diffusion models
2. **Data Preparation**: Load pre-split augmented data (train/val sets)
3. **Sampling and Refinement**: Use diffusion model to sample candidates and refine predictions using the forward model.
4. **Visualization**: Plot results comparing before and after refinement

#### Data Pipeline Consistency:
- **RNN Augmentation** → Augment data, split into train/val, fit scalers → Save `combined_augmented_train.xlsx`, `val.xlsx`, `scaler_x_new.pkl`, `scaler_y_new.pkl`
- **Diffusion Model** → Load pre-split data and scalers for training
- **Forward Model** → Load pre-split data and scalers for training
- **Prediction (this notebook)** → Load pre-split data and scalers for evaluation

### Environment Settings

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib, pandas as pd

In [ ]:
import matplotlib as mpl
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['xtick.major.width'] = 3
mpl.rcParams['ytick.major.width'] = 3

In [ ]:
# Use serif + bold global settings before drawing
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

### 1. Data and Models Loading and Preprocessing

#### 1.1 Models Loading

In [ ]:
# Load Models
base_dir = Path("")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Forward Model
class ImprovedRegressionModel(nn.Module):
    def __init__(self,
                 moduli_dim: int = 6,
                 dist_dim: int = 2,
                 hidden_dim: int = 256,
                 trunk_layers: int = 2,
                 dropout: float = 0.1):
        super().__init__()
        assert dist_dim == 2, "This model expects dist_dim=2"
        self.moduli_dim = moduli_dim
        self.dist_dim   = dist_dim

        trunk = []
        in_dim = moduli_dim
        for _ in range(trunk_layers):
            trunk.append(nn.Linear(in_dim, hidden_dim))
            trunk.append(nn.LayerNorm(hidden_dim))
            trunk.append(nn.GELU())
            trunk.append(nn.Dropout(dropout))
            in_dim = hidden_dim
        self.trunk = nn.Sequential(*trunk)

        self.dist_embed = nn.Sequential(
            nn.Linear(dist_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        fusion_dim = hidden_dim + (hidden_dim // 4)

        self.head0 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.head1 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1)
        )

        self.log_sigma0 = nn.Parameter(torch.zeros(()))
        self.log_sigma1 = nn.Parameter(torch.zeros(()))

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_mod  = x[:, : self.moduli_dim]
        x_dist = x[:, self.moduli_dim:]

        z_mod = self.trunk(x_mod)
        z_dist = self.dist_embed(x_dist)

        z = torch.cat([z_mod, z_dist], dim=1)

        y0 = self.head0(z)
        y1 = self.head1(z)

        return torch.cat([y0, y1], dim=-1)

forward_model = ImprovedRegressionModel(moduli_dim=6, dist_dim=2, hidden_dim=256, trunk_layers=2, dropout=0.1).to(device)
forward_model.load_state_dict(torch.load(base_dir/"artifacts/soft_material/forward_model_weights_new.pt", map_location=device))
forward_model.eval()

In [ ]:
# Load Diffusion Model
class DiffusionSchedule(nn.Module):
    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, eps: float = 1e-8):
        super().__init__()
        self.T = T
        self.eps = eps

        beta = torch.linspace(beta_start, beta_end, T)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]], dim=0)

        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0[:, 6:])

        ab = self.alpha_bar.to(x0.device)[t].clamp(self.eps, 1.0 - self.eps)
        sqrt_ab = ab.sqrt().view(-1, 1)
        sqrt_omb = (1.0 - ab).sqrt().view(-1, 1)

        x_mod = x0[:, :6]
        x_dist = x0[:, 6:]
        x_dist_t = sqrt_ab * x_dist + sqrt_omb * noise

        return torch.cat([x_mod, x_dist_t], dim=1)

class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        half = dim // 2
        factor = math.log(10000.0) / (half - 1)
        self.register_buffer('factor', torch.tensor(factor))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        emb_scale = torch.exp(-self.factor * torch.arange(half, device=t.device))
        emb = t[:, None].float() * emb_scale[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=1)

class EnhancedDiffusionModel(nn.Module):
    """Enhanced diffusion model: deeper network with better regularization"""
    def __init__(self, x_dim=8, y_dim=2, time_dim=32, hidden_dim=256):
        super().__init__()
        self.x_dim = x_dim

        # === Moduli branch (Ep1-Epp3): enhanced capacity ===
        self.moduli_embed = nn.Sequential(
            nn.Linear(6, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Distribution branch (t1, t2): deep MLP ===
        self.dist_mlp = nn.Sequential(
            nn.Linear(2, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Y conditioning branch (EpH, EppH) ===
        self.y_mlp = nn.Sequential(
            nn.Linear(y_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128)
        )

        # === Time embedding branch ===
        self.time_embed = TimeEmbedding(time_dim)
        self.time_fc = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

        # === Fusion layer: deep network ===
        fusion_dim = 128 + 128 + 128 + 128  # moduli + dist + y + time
        self.fusion_fc = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)  # Output noise prediction (t1, t2)
        )

    def forward(self, x, t, y):
        """Forward propagation
        x: [B, 8] - [Ep1, Ep2, Ep3, Epp1, Epp2, Epp3, t1_noisy, t2_noisy]
        t: [B] - timestep
        y: [B, 2] - condition [EpH, EppH]
        """
        x_mod = x[:, :6]   # Ep1-Epp3
        x_dist = x[:, 6:]  # t1, t2 (noisy)

        # Feature extraction from each branch
        mod_feat = self.moduli_embed(x_mod)
        dist_feat = self.dist_mlp(x_dist)
        y_feat = self.y_mlp(y)
        t_emb = self.time_embed(t)
        t_feat = self.time_fc(t_emb)

        # Fuse all features
        fused = torch.cat([mod_feat, dist_feat, y_feat, t_feat], dim=1)
        eps_pred = self.fusion_fc(fused)

        return eps_pred

diffusion_model = EnhancedDiffusionModel(x_dim=8, y_dim=2, time_dim=32, hidden_dim=384).to(device)
state_dict = torch.load(base_dir/"artifacts/soft_material/diffusion_model_new_best.pt", map_location=device)
diffusion_model.load_state_dict(state_dict)
diffusion_model.eval()

#### 1.2 Data Preparation

In [ ]:
# Data Preparation
import joblib, pandas as pd

# Load new scalers (fitted on augmented training set)
scaler_x = joblib.load("artifacts/soft_material/scaler_x_new.pkl")
scaler_y = joblib.load("artifacts/soft_material/scaler_y_new.pkl")

print("Loaded scalers:")
print(f"  - artifacts/soft_material/scaler_x_new.pkl")
print(f"  - artifacts/soft_material/scaler_y_new.pkl")

# Load pre-split augmented data (consistent with diffusion and forward model training)
df_train = pd.read_excel("data/combined_augmented_train.xlsx", header=0)
df_val = pd.read_excel("data/val.xlsx", header=0)

X_tr = df_train.iloc[:, :8].values
Y_tr = df_train.iloc[:, 8:].values
X_val = df_val.iloc[:, :8].values
Y_val = df_val.iloc[:, 8:].values

print(f"\nLoaded pre-split data:")
print(f"  - Training set: {len(X_tr)} samples (from combined_augmented_train.xlsx)")
print(f"  - Validation set: {len(X_val)} samples (from val.xlsx)")

# Verify group separation (for information only)
MOD = ['Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3']
groups_train = pd.util.hash_pandas_object(df_train[MOD].round(6), index=False).astype(np.int64).to_numpy()
groups_val = pd.util.hash_pandas_object(df_val[MOD].round(6), index=False).astype(np.int64).to_numpy()

unique_train = len(np.unique(groups_train))
unique_val = len(np.unique(groups_val))

print(f"\nGroup information:")
print(f"  - Training groups: {unique_train}")
print(f"  - Validation groups: {unique_val}")

# Check for data leakage
overlap = set(np.unique(groups_train)) & set(np.unique(groups_val))
if overlap:
    print(f"  Warning: Found {len(overlap)} overlapping groups between train and val!")
else:
    print(f"  ✓ No data leakage: Training and validation groups are completely separate")

print(f"\n✓ Data preparation complete - using same pre-split data as diffusion and forward models")

#### 1.3 Diffusion Schedule

In [ ]:
# Diffusion Schedule
schedule = DiffusionSchedule(T=1000).to(device)

# Sampling Function
@torch.no_grad()
def sample_with_fixed_moduli(diffusion_model, schedule, y_cond, moduli_fixed):
    x_dist = torch.randn(1, 2, device=device)
    x = torch.cat([moduli_fixed, x_dist], dim=1)

    for t in reversed(range(schedule.T)):
        t_tensor = torch.full((1,), t, device=device, dtype=torch.long)
        eps_dist = diffusion_model(x, t_tensor, y_cond)

        beta_t = schedule.beta[t]
        alpha_t = schedule.alpha[t]
        alpha_bar_t = schedule.alpha_bar[t]
        denom_sqrt = torch.sqrt((1 - alpha_bar_t).clamp(min=schedule.eps))

        x_mod_part = moduli_fixed
        x_dist_part = x[:, 6:]

        mu_dist = (1 / torch.sqrt(alpha_t.clamp(min=schedule.eps))) * (
            x_dist_part - (beta_t / denom_sqrt) * eps_dist
        )

        if t > 0:
            ab_prev = schedule.alpha_bar_prev[t]
            sigma_t = torch.sqrt(beta_t * (1 - ab_prev) / (1 - alpha_bar_t))
            z = torch.randn_like(mu_dist)
            x_dist_prev = mu_dist + sigma_t * z
        else:
            x_dist_prev = mu_dist

        x = torch.cat([x_mod_part, x_dist_prev], dim=1)

    return x

### 2. Prediction and refinement

#### 2.1 Sampe loading

In [ ]:
# Extract Fixed Moduli (use validation set for prediction)
# Select a sample from validation set to ensure it's from a different group
val_sample_idx = 2593  # Choose a sample from validation set
fixed_moduli_raw = X_val[val_sample_idx, :6].reshape(1, 6)

# Prepare for Scaling
moduli_tmp = np.zeros((1, 8))
moduli_tmp[0, :6] = fixed_moduli_raw
fixed_moduli_scaled_full = scaler_x.transform(moduli_tmp)
fixed_moduli_scaled = fixed_moduli_scaled_full[:, :6]

# Convert to Tensor
moduli_fixed = torch.tensor(fixed_moduli_scaled, dtype=torch.float32, device=device)
print(f"Selected validation sample {val_sample_idx}:")
print(fixed_moduli_raw)

In [ ]:
# Prepare Target Y
# raw_target = Y_val[val_sample_idx, -2:].reshape(1, 2)
raw_target = np.array([[3.5,1.2]])
std_target = scaler_y.transform(raw_target)
target_y = torch.tensor(std_target, dtype=torch.float32, device=device)
print(raw_target)

#### 2.2 Sampling and prediction

In [ ]:
# Sampling and Prediction
num_samples = 250
cands_x_std = []
preds_std = []

for _ in range(num_samples):
    x_gen = sample_with_fixed_moduli(diffusion_model, schedule, target_y, moduli_fixed)
    x_gen_np = x_gen.cpu().numpy().reshape(-1)
    cands_x_std.append(x_gen_np)

    y_pred = forward_model(x_gen)
    y_pred_np = y_pred.detach().cpu().numpy().reshape(-1)
    preds_std.append(y_pred_np)

cands_x_std = np.vstack(cands_x_std)
preds_std = np.vstack(preds_std)

# Inverse Transform
cands_x_orig = scaler_x.inverse_transform(cands_x_std)
preds_orig = scaler_y.inverse_transform(preds_std)

# Calculate Errors
err0 = np.abs(preds_orig[:, 0] - raw_target[0, 0])
err1 = np.abs(preds_orig[:, 1] - raw_target[0, 1])

# Pareto Frontier
def pareto_frontier(e0, e1):
    is_pareto = np.ones(e0.shape[0], dtype=bool)
    for i, (a, b) in enumerate(zip(e0, e1)):
        dominated = (e0 <= a) & (e1 <= b) & ((e0 < a) | (e1 < b))
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

mask_pareto = pareto_frontier(err0, err1)
pareto_x_orig = cands_x_orig[mask_pareto]
pareto_y_orig = preds_orig[mask_pareto]
pareto_errs = np.stack([err0, err1], axis=1)[mask_pareto]

print(f"Total candidates: {num_samples}")
print(f"Pareto front size: {pareto_x_orig.shape[0]}")

#### 2.3 Refinement

In [ ]:
# Refinement
errors_orig = np.mean((preds_orig - raw_target)**2, axis=1)
best_idx = np.argmin(errors_orig)

best_x_std_np = cands_x_std[best_idx:best_idx+1]
best_y_std_np = preds_std[best_idx:best_idx+1]

# Use the original moduli_fixed from Cell 11 (already in torch tensor format)
# Don't extract from candidates to avoid inconsistency
dist_initial_np = best_x_std_np[:, 6:]

dist_var = torch.tensor(dist_initial_np, dtype=torch.float32, device=device, requires_grad=True)

opt_refine = torch.optim.Adam([dist_var], lr=1e-2)
criterion = nn.MSELoss()

std_target = torch.tensor(
    scaler_y.transform(raw_target),
    dtype=torch.float32,
    device=device
)

num_steps = 100
snap_steps = list(range(0, num_steps, 10))
snapshot_steps = []
snapshot_preds = []

for step in range(num_steps):
    opt_refine.zero_grad()
    x_refine = torch.cat([moduli_fixed, dist_var], dim=1)
    y_hat = forward_model(x_refine)
    loss = nn.MSELoss()(y_hat, std_target)
    loss.backward()
    opt_refine.step()

    if step in snap_steps:
        snapshot_steps.append(step)
        with torch.no_grad():
            pred_np = forward_model(
                torch.cat([moduli_fixed, dist_var], dim=1)
            ).detach().cpu().numpy().flatten()
        snapshot_preds.append(pred_np)

snapshot_preds = np.array(snapshot_preds)

orig_x_orig = cands_x_orig[best_idx]
orig_y_orig = preds_orig[best_idx]
refined_dist_std = dist_var.detach().cpu().numpy()

# Use the original moduli_fixed (convert to numpy for concatenation)
moduli_fixed_np = moduli_fixed.detach().cpu().numpy()
refined_x_std = np.hstack([moduli_fixed_np, refined_dist_std])
refined_y_std = forward_model(torch.tensor(refined_x_std, dtype=torch.float32, device=device)).detach().cpu().numpy()

refined_x_orig = scaler_x.inverse_transform(refined_x_std)
refined_y_orig = scaler_y.inverse_transform(refined_y_std)

refined_mse = np.mean((refined_y_orig - raw_target)**2)

print("Before refinement:")
print(f"  x = {np.round(orig_x_orig, 4)}")
print(f"  ŷ = {np.round(orig_y_orig, 4)}")
print(f"  MSE = {errors_orig[best_idx]:.6f}\n")

print("After refinement:")
print(f"  x = {np.round(refined_x_orig.flatten(), 4)}")
print(f"  ŷ = {np.round(refined_y_orig.flatten(), 4)}")
print(f"  MSE = {refined_mse:.6f}")


#### 2.4 Plot

In [ ]:
# Plot and save scatter plots for this sample
top_idx = np.argsort(errors_orig)[:50]

# Dynamically set x-axis range based on raw_target
x_val = raw_target[0, 0]
if 1 <= x_val <= 5.5:
    # Take the rounded integer and then add/subtract 1
    x_min = round(x_val) - 1
    x_max = round(x_val) + 1
elif 0 <= x_val <= 1:
    x_min, x_max = -0.5, 1.5
elif 5.5 <= x_val <= 6.0:
    x_min, x_max = 4.5, 6.5

# Dynamically set y-axis range based on raw_target
y_val = raw_target[0, 1]
if 0 <= y_val < 1.8:
    y_min, y_max = 0, 2
elif 1.8 <= y_val <= 2.5:
    y_min, y_max = 0.5, 2.5

plt.figure(figsize=(10, 10))
# Top 50 candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=50, label='Candidates')
# Before refinement
plt.scatter(orig_y_orig[0], orig_y_orig[1], c='blue', s=100, label='Before refinement', edgecolors='b')
# After refinement
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='green', s=100, label='After refinement', marker='D')
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200, label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')

plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.legend(prop={'size':22, 'weight':'bold'})

# Thicken the axes spines
ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.5))
ax.yaxis.set_major_locator(MultipleLocator(base=0.5)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for Candidates with Refinement
plt.figure(figsize=(6, 6))
# Candidates
plt.scatter(preds_orig[top_idx, 0], preds_orig[top_idx, 1], c='gray', alpha=0.5, s=100)
# Before refinement
plt.scatter(orig_y_orig[0], orig_y_orig[1], c='blue', s=200)
# After refinement
plt.scatter(refined_y_orig[0, 0], refined_y_orig[0, 1], c='green', s=200)
# Target
plt.scatter(raw_target[0, 0], raw_target[0, 1], c='red', marker='x', s=200)

plt.xticks([])
plt.yticks([])

ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

In [ ]:
# Visualization
snapshot_preds_std = np.array(snapshot_preds)
snapshot_preds_orig = scaler_y.inverse_transform(snapshot_preds_std)

# Plot Iteration Path
plt.figure(figsize=(10,10))
n = len(snapshot_preds_orig)
plt.plot(snapshot_preds_orig[:, 0], snapshot_preds_orig[:, 1], '--o', markevery=range(n-1), label='Iteration Path', linewidth=3, markersize=8, markeredgewidth=2, alpha=0.6, color='red')
plt.scatter(snapshot_preds_orig[-1, 0], snapshot_preds_orig[-1, 1], s=200, edgecolor='black', facecolor='yellow', linewidth=2, label='After refinement')
plt.scatter([raw_target[0, 0]], [raw_target[0, 1]], c='red', marker='*', s=300, linewidth=2, edgecolor='black', label=f'Target ({raw_target[0,0]:.3f}, {raw_target[0,1]:.3f})')
plt.xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=6)
plt.legend(prop={'size':22, 'weight':'bold', 'family':'serif'}, frameon=True, loc='upper left')
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.axis('equal')

ax = plt.gca()
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.002))
ax.yaxis.set_major_locator(MultipleLocator(base=0.002)) 

for spine in ax.spines.values():
        spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Zoomed-in Detail for Iteration Path
plt.figure(figsize=(6, 6))
# Last 5 steps of iteration path
plt.plot(snapshot_preds_orig[-5:, 0], snapshot_preds_orig[-5:, 1], '--o', markevery=range(4), linewidth=3, markersize=8, markeredgewidth=2, alpha=0.6, color='red')
# Final point
plt.scatter(snapshot_preds_orig[-1, 0], snapshot_preds_orig[-1, 1], s=200, edgecolor='black', facecolor='yellow', linewidth=2)
# Target
plt.scatter([raw_target[0, 0]], [raw_target[0, 1]], c='red', marker='*', s=300, linewidth=2, edgecolor='black')

plt.xticks([])
plt.yticks([])
plt.axis('equal')

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
        spine.set_linewidth(4.5)
        spine.set_linestyle('--') 

plt.tight_layout()
plt.show()

#### 2.5 Save candidates

In [ ]:
# Save complete candidates data to Excel
import pandas as pd
import numpy as np

# Prepare feature names
feature_names = ['Ep1','Ep2','Ep3','Epp1','Epp2','Epp3','t1','t2']

# Create comprehensive DataFrame with all candidates
complete_candidates_df = pd.DataFrame(
    np.hstack([
        cands_x_orig,           # All input features (Ep1-Epp3, t1, t2)
        preds_orig,             # All predictions (yhat_0, yhat_1)
        errors_orig[:, None],   # MSE errors
        err0[:, None],          # Error in dimension 0
        err1[:, None]           # Error in dimension 1
    ]),
    columns=[
        *feature_names,         # Ep1, Ep2, Ep3, Epp1, Epp2, Epp3, t1, t2
        "yhat_0", "yhat_1",     # Predicted values
        "mse_to_target",        # Mean squared error to target
        "err_dim0", "err_dim1"  # Individual dimension errors
    ]
)

# Add target values for reference
complete_candidates_df["target_0"] = raw_target[0, 0]
complete_candidates_df["target_1"] = raw_target[0, 1]

# Add Pareto front indicator
complete_candidates_df["is_pareto"] = mask_pareto

# Save to Excel
output_file = "artifacts/soft_material/complete_candidates_250.xlsx"
complete_candidates_df.to_excel(output_file, index=False)

print(f"Saved complete candidates data to: {output_file}")
print(f"Total candidates: {len(complete_candidates_df)}")
print(f"Columns: {list(complete_candidates_df.columns)}")
print(f"Data shape: {complete_candidates_df.shape}")

# Display basic statistics
print(f"\nBasic statistics:")
print(f"t1 range: {complete_candidates_df['t1'].min():.4f} - {complete_candidates_df['t1'].max():.4f}")
print(f"t2 range: {complete_candidates_df['t2'].min():.4f} - {complete_candidates_df['t2'].max():.4f}")
print(f"MSE range: {complete_candidates_df['mse_to_target'].min():.6f} - {complete_candidates_df['mse_to_target'].max():.6f}")
print(f"Pareto front size: {complete_candidates_df['is_pareto'].sum()}")


### 3. Evaluation & Filtering

In [ ]:
# Load complete candidates data from Excel
import pandas as pd
import numpy as np

# Load the saved complete candidates data
input_file = "artifacts/soft_material/complete_candidates_250.xlsx"
loaded_candidates_df = pd.read_excel(input_file)

# Extract the data for evaluation
feature_names = ['Ep1','Ep2','Ep3','Epp1','Epp2','Epp3','t1','t2']

# Extract input features (x values)
cands_x_orig_loaded = loaded_candidates_df[feature_names].values

# Extract predictions (y values)
preds_orig_loaded = loaded_candidates_df[["yhat_0", "yhat_1"]].values

# Extract errors
errors_orig_loaded = loaded_candidates_df["mse_to_target"].values
err0_loaded = loaded_candidates_df["err_dim0"].values
err1_loaded = loaded_candidates_df["err_dim1"].values

# Extract target values
raw_target_loaded = loaded_candidates_df[["target_0", "target_1"]].values[0:1]  # Keep as 2D array

# Extract Pareto front mask
mask_pareto_loaded = loaded_candidates_df["is_pareto"].values.astype(bool)

# Update variables for use in subsequent cells
cands_x_orig = cands_x_orig_loaded
preds_orig = preds_orig_loaded
errors_orig = errors_orig_loaded
err0 = err0_loaded
err1 = err1_loaded
raw_target = raw_target_loaded
mask_pareto = mask_pareto_loaded

print(f"✓ Variables updated for evaluation. Ready to proceed with filtering and analysis.")


In [ ]:
# =========================================================
# [Add-on] Inverse candidates: evaluation & filtering
#   Paste this block at the end of your current script
#   Variables already computed above are reused:
#   - cands_x_std, cands_x_orig, preds_std, preds_orig
#   - raw_target, errors_orig, err0, err1, mask_pareto
# =========================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.size": 20,
    "font.weight": "bold",
    "font.family": "serif"
})

def beautify_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    ax.tick_params(axis='both', which='major', length=8, width=3)

feature_names = ['Ep1','Ep2','Ep3','Epp1','Epp2','Epp3','t1','t2']

In [ ]:
# ---- 1) Error summary & Top-k selection ----
ERR_METRIC = "MSE"
TOP_K = 15

err_all = errors_orig.copy()
order = np.argsort(err_all)
top_idx = order[:TOP_K]

print("\n=== Inverse candidates: summary ===")
print(f"Total candidates = {len(err_all)}")
print(f"Error mean={err_all.mean():.4f}, std={err_all.std():.4f}, "
      f"min={err_all.min():.4f}, median={np.median(err_all):.4f}")
print(f"Top-{TOP_K} (by {ERR_METRIC} to target):")
for r, i in enumerate(top_idx, 1):
    xrow = np.round(cands_x_orig[i], 4)
    yrow = np.round(preds_orig[i], 4)
    print(f"#{r:02d}  err={err_all[i]:.4f}   x={xrow}   y_hat={yrow}")

# Save top-k to CSV
top_df = pd.DataFrame(
    np.hstack([cands_x_orig[top_idx], preds_orig[top_idx], err_all[top_idx, None]]),
    columns=[*feature_names, "yhat_0", "yhat_1", "mse_to_target"]
)
top_df["target_0"] = raw_target[0,0]
top_df["target_1"] = raw_target[0,1]
top_df.to_csv("artifacts/soft_material/inverse_topk_candidates.csv", index=False)
print("Saved: inverse_topk_candidates.csv")

In [ ]:
# ---- 2) Multi-solution distributions (t1, t2) ----
fig, ax = plt.subplots(figsize=(6,8))
ax.hist(cands_x_orig[:, 6], bins=30, alpha=0.9, edgecolor='black')
ax.set_xlabel("$t_1$", fontsize=26, fontweight='bold', fontfamily='serif', labelpad=10); 
ax.set_ylabel('Candidates Count', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6); 
beautify_ax(ax)
from matplotlib.ticker import MultipleLocator
ax.yaxis.set_major_locator(MultipleLocator(base=5)) 
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6,8))
ax.hist(cands_x_orig[:, 7], bins=30, alpha=0.9, edgecolor='black')
ax.set_xlabel("$t_2$", fontsize=26, fontweight='bold', fontfamily='serif', labelpad=10); 
ax.set_ylabel('Candidates Count', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6);  
beautify_ax(ax)
from matplotlib.ticker import MultipleLocator
ax.yaxis.set_major_locator(MultipleLocator(base=5)) 
plt.tight_layout(); plt.show()

In [ ]:
# ---- 3) Pareto scatter (|err0| vs |err1|) ----
fig, ax = plt.subplots(figsize=(8,8))

# Get Top 50 candidates based on MSE
top_50_indices = np.argsort(errors_orig)[:50]

# Plot only Top 50 candidates
ax.scatter(err0[top_50_indices], err1[top_50_indices], s=30, alpha=0.6, label='All Candidates')

# Plot Pareto front from Top 50 (if any)
top_50_pareto_mask = mask_pareto[top_50_indices]
if np.any(top_50_pareto_mask):
    pareto_indices = top_50_indices[top_50_pareto_mask]
    ax.scatter(err0[pareto_indices], err1[pareto_indices],
               s=80, marker='D', label='Pareto Front\n Candidates')

ax.set_xlabel('Abs. error in $E_H^{\prime}$ (MPa)', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=10); 
ax.set_ylabel('Abs. error in $E_H^{\prime\prime}$ (MPa)', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6); 
ax.legend()
beautify_ax(ax)
from matplotlib.ticker import MultipleLocator
ax.xaxis.set_major_locator(MultipleLocator(base=0.02)) 
ax.yaxis.set_major_locator(MultipleLocator(base=0.02)) 
plt.tight_layout(); plt.show()

In [ ]:
# ---- 4) (Optional) Diversity pick ----
MIN_DIST = 0.05
picked = []
for i in top_idx:
    t1i, t2i = cands_x_orig[i, 6], cands_x_orig[i, 7]
    if all(np.hypot(t1i - cands_x_orig[j, 6], t2i - cands_x_orig[j, 7]) >= MIN_DIST for j in picked):
        picked.append(i)
picked = np.array(picked, dtype=int)
print(f"\nDiversity-filtered picks: {len(picked)}/{TOP_K} kept (threshold={MIN_DIST})")
print("Indices:", picked.tolist())

pd.DataFrame(
    np.hstack([cands_x_orig[picked], preds_orig[picked], err_all[picked, None]]),
    columns=[*feature_names, "yhat_0", "yhat_1", "mse_to_target"]
).to_csv("artifacts/soft_material/inverse_topk_diverse.csv", index=False)
print("Saved: inverse_topk_diverse.csv")